# Customer Churn Analysis

**Goal:** Analyze customer subscription data to understand churn behavior, calculate key business KPIs (churn rate, retention rate, ARPU, revenue at risk), and identify the factors most associated with churn (plan type, state, support escalations).

**Data sources:** Three raw tables (Customer, Subscription, Support) loaded from a single multi-sheet Excel file, cleaned and merged into a single analysis-ready dataset.

**Tools:** Python (Pandas, NumPy), Matplotlib, Seaborn

**Workflow:**
1. Data loading (multi-sheet Excel)
2. Data cleaning (per table)
3. Feature engineering (churn flag, tenure, churn risk bucket)
4. Merging into a single dataset
5. KPI calculation
6. Visualization (Matplotlib & Seaborn)
7. Pivot table summaries


## 1. Imports

In [ ]:
# Core data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Data Loading
The raw data is stored as multiple sheets in a single Excel workbook. Each sheet is loaded into its own DataFrame, named dynamically as `df_<sheet_name>`.

In [ ]:
# Load every sheet of the raw Excel file into its own DataFrame.
# Sheet 'db_customer' -> df_db_customer, 'db_subscription' -> df_db_subscription, etc.
excel_file = 'customer_churn_data_raw.xlsx'
excel_data = pd.ExcelFile(excel_file)

for sheet in excel_data.sheet_names:
    df = pd.read_excel(excel_data, sheet_name=sheet)
    globals()[f"df_{sheet}"] = df
    print(f"Created dataframe: df_{sheet}")


## 3. Data Cleaning — Customer Table

In [ ]:
# Quick look at the first and last rows to understand structure and spot obvious issues
df_db_customer.head()

In [ ]:
df_db_customer.tail()

In [ ]:
# Check dtypes and null counts before cleaning
df_db_customer.info()

In [ ]:
# Rename for clarity/consistency with the rest of the pipeline
df_db_customer.rename(columns={'name': 'customer_name'}, inplace=True)

In [ ]:
# Drop columns that aren't needed for this analysis
df_db_customer.drop(columns=['interests', 'pincode'], inplace=True)

In [ ]:
# Convert date-of-birth from string to proper datetime for future age/date calculations
df_db_customer['dob'] = pd.to_datetime(df_db_customer['dob'])

In [ ]:
# Confirm dtype fix and check remaining nulls
df_db_customer.info()

In [ ]:
# Inspect inconsistent category labels in 'gender'
df_db_customer['gender'].unique()

In [ ]:
# Standardize gender labels ('Men'/'Women' -> 'Male'/'Female')
df_db_customer['gender'] = df_db_customer['gender'].replace({'Men': 'Male', 'Women': 'Female'})

In [ ]:
# Identify rows where 'country' is missing, to decide how to fill them
df_db_customer[df_db_customer['country'].isna()]

In [ ]:
# Fill missing 'country' values using a state -> country lookup built from rows
# that already have a known country for that state.
state_country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()
df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))


In [ ]:
# Confirm the customer table is now clean
df_db_customer

## 4. Data Cleaning — Subscription Table

In [ ]:
df_db_subscription.head()

In [ ]:
df_db_subscription.tail()

In [ ]:
df_db_subscription.info()

In [ ]:
# Convert all date columns from string to datetime in one pass
date_col = ['subscription_start_date', 'cancellation_date', 'renewal_date']
df_db_subscription[date_col] = df_db_subscription[date_col].apply(pd.to_datetime)


In [ ]:
# Confirm date columns are now proper datetime dtype
df_db_subscription.info()

## 5. Data Cleaning — Support Table

In [ ]:
df_db_support.head()

In [ ]:
df_db_support.tail()

In [ ]:
df_db_support.info()

In [ ]:
# Drop an unlabeled/unused column and free-text comments (not needed for quantitative analysis)
df_db_support.drop(columns=['col_1', 'comment'], inplace=True)

In [ ]:
df_db_support.info()

## 6. Feature Engineering — Churn Flag
A customer is considered **churned** if they have a non-null cancellation date.

In [ ]:
# churn_flag = 1 if the customer has a cancellation date, else 0 (still active)
df_db_subscription['churn_flag'] = np.where(df_db_subscription['cancellation_date'].notna(), 1, 0)


In [ ]:
df_db_subscription.head()

## 7. Merging Datasets
Subscription is treated as the base table (one row per subscription); customer and support details are joined in via `customerid`.

In [ ]:
df = (df_db_subscription
      .merge(df_db_customer, on='customerid', how='left')
      .merge(df_db_support, on='customerid', how='left'))


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.shape

## 8. Deduplicating Support Records
The support table has multiple complaint rows per customer. Before merging again, count total complaints per customer and keep only the most recent complaint row so the merge doesn't duplicate subscription rows.

In [ ]:
# Total number of complaints raised by each customer, kept as its own column
df_db_support['complain_count'] = df_db_support.groupby('customerid')['customerid'].transform('count')

In [ ]:
# Keep only the latest complaint per customer so the join stays one-to-one
df_db_support = df_db_support.sort_values('complaint_date').drop_duplicates('customerid', keep='last')

In [ ]:
df_db_support.shape

In [ ]:
# Re-run the merge now that support is deduplicated -> one row per subscription
df = (df_db_subscription
      .merge(df_db_customer, on='customerid', how='left')
      .merge(df_db_support, on='customerid', how='left'))


In [ ]:
df.shape

In [ ]:
# Export the cleaned, merged dataset for reuse / sharing
df.to_csv('exported_customer_churn_data.csv', index=False)

## 9. Key Business Metrics (KPIs)

In [ ]:
# Overall churn rate (% of subscriptions that were cancelled)
churn_rate = df['churn_flag'].mean() * 100
round(churn_rate, 2)

In [ ]:
# Retention Rate
retention_rate = 100 - churn_rate
round(retention_rate, 2)

In [ ]:
# Churn by plan type
churn_by_plan = df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index(name='churn_rate_percent')
churn_by_plan

In [ ]:
# Churn by state - sum(revenue) & count of churned users, to see where revenue loss is concentrated
churn_by_state = (df[df['churn_flag'] == 1]
                  .groupby('state')
                  .agg(revenue=('monthly_charges', 'sum'), customer_count=('customerid', 'nunique'))
                  .reset_index())
churn_by_state

In [ ]:
# Churn by subscription type - sum(revenue) & count of churned users
churn_by_subscription = (df[df['churn_flag'] == 1]
                          .groupby('subscription_type')
                          .agg(revenue=('monthly_charges', 'sum'), customer_count=('customerid', 'nunique'))
                          .reset_index())
churn_by_subscription

In [ ]:
# Average Revenue Per User (ARPU)
arpu = round(df['monthly_charges'].mean(), 2)
arpu

In [ ]:
# Average customer tenure in days.
# For churned customers: cancellation_date - start_date.
# For active customers: today - start_date (tenure so far).
today = pd.Timestamp.today()
df['tenure_days'] = np.where(
    df['cancellation_date'].notna(),
    (df['cancellation_date'] - df['subscription_start_date']).dt.days,
    (today - df['subscription_start_date']).dt.days
)
avg_tenure = round(df['tenure_days'].mean())
avg_tenure


In [ ]:
# Revenue at risk - monthly revenue lost from customers who have already churned
revenue_at_risk = df.loc[df['churn_flag'] == 1, 'monthly_charges'].sum()
revenue_at_risk

In [ ]:
# Escalation rate - % of customers whose support ticket was escalated
escalation_rate = (df['escalations'] == 'Y').mean() * 100
round(escalation_rate, 2)

In [ ]:
# Average number of complaints per unique customer
avg_complain = df['complain_count'].sum() / df['customerid'].nunique()
round(avg_complain, 2)

In [ ]:
# Correlation between support escalation and churn.
# 'escalations' is recoded from Y/N to 1/0 so it can be correlated numerically with churn_flag.
corr_df = df[['escalations', 'churn_flag']].dropna()
df['escalations'] = np.where(df['escalations'] == 'Y', 1, 0)  # encoding string to int type
correlation = corr_df['escalations'].corr(df['churn_flag'])
round(correlation, 2)


In [ ]:
# Bucket each customer into a churn-risk tier based on their churn_score
condition = [
    (df['churn_score'] < 50),
    (df['churn_score'] >= 50) & (df['churn_score'] < 70),
    (df['churn_score'] >= 70)
]
choice = ['low', 'med', 'high']
df['churn_risk'] = np.select(condition, choice, default='unknown')


In [ ]:
df.head()

## 10. Visualization — Matplotlib

In [ ]:
# Work on a copy so raw KPI columns above stay untouched by any visualization-only tweaks
df_visual = df.copy()

In [ ]:
# Monthly churn trend: how many customers cancelled each month
df_visual['cancellation_month'] = df_visual['cancellation_date'].dt.to_period('M')
churn_trend = df_visual[df_visual['churn_flag'] == 1].groupby('cancellation_month').size()

plt.figure(figsize=(5, 3))
plt.title('Monthly Churn Trend')
plt.xlabel('Month')
plt.ylabel('Churned Customers')
plt.plot(churn_trend.index.astype(str), churn_trend.values, color='red', marker='o')
plt.show()


In [ ]:
# Churn rate by plan type
churn_plan = df_visual.groupby('plan_type')['churn_flag'].mean()

plt.figure(figsize=(5, 3))
plt.bar(churn_plan.index, churn_plan.values)
plt.xlabel("Plan")
plt.ylabel("Churn Rate")
plt.title("Customer Churn by Plan")
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Churn rate by state
churn_plan = df_visual.groupby('state')['churn_flag'].mean()

colors = plt.cm.Set2(np.linspace(0, 1, len(churn_plan)))
plt.figure(figsize=(12, 5))
plt.bar(churn_plan.index, churn_plan.values, color=colors)
plt.title("Customer Churn by State")
plt.xticks(rotation=45)
plt.show()


## 11. Visualization — Seaborn

In [ ]:
# Correlation heatmap across all numeric columns
sns.heatmap(df_visual.corr(numeric_only=True), annot=True)

In [ ]:
df_visual.columns

In [ ]:
# Preview the categorical columns we'll encode for correlation analysis
df_visual[['plan_type', 'contract_type', 'churn_score', 'churn_flag', 'churn_risk', 'escalations']].head()

In [ ]:
# Ordinal-encode categorical columns so they can be included in a numeric correlation matrix.
# Order matters here (e.g. Basic < Standard < Premium, low < med < high risk).
df_encoded = df_visual[['plan_type', 'contract_type', 'churn_score', 'churn_flag', 'churn_risk', 'escalations']].copy()

order_mappings = {
    'plan_type': ['Basic', 'Standard', 'Premium'],
    'contract_type': ['Monthly', 'Annual'],
    'churn_risk': ['low', 'med', 'high']
}

for col, order in order_mappings.items():
    df_encoded[col] = pd.Categorical(df_encoded[col].astype('category'), categories=order, ordered=True).codes


In [ ]:
df_encoded.head()

In [ ]:
# Correlation heatmap on the encoded (fully numeric) feature set
sns.heatmap(df_encoded.corr(), annot=True)

In [ ]:
# Pairplot - visualize pairwise relationships across all encoded features at once
sns.pairplot(df_encoded)

In [ ]:
# Multi-dimensional comparison: monthly charges by plan type, split by gender and churn risk tier
sns.catplot(
    data=df_visual,
    x='plan_type',
    y='monthly_charges',
    hue='gender',
    col='churn_risk'
)


## 12. Pivot Table Summaries

In [ ]:
# Churn rate by plan type, in pivot-table form
pd.pivot_table(
    df_visual,
    values='churn_flag',
    index='plan_type',
    aggfunc='mean'
).reset_index()

In [ ]:
# Consolidated summary: churn rate, total revenue, and unique customer count per plan type
pd.pivot_table(
    df_visual,
    index='plan_type',
    values=['monthly_charges', 'customerid', 'churn_flag'],
    aggfunc={
        'monthly_charges': 'sum',
        'customerid': 'nunique',
        'churn_flag': 'mean'
    }
)


## Key Takeaways

- Overall churn rate, retention rate, ARPU, and revenue-at-risk give a quick health check of the subscriber base.
- Churn varies meaningfully by **plan type** and **state**, pointing to where retention efforts should be focused.
- Support escalations show a measurable correlation with churn, suggesting proactive support quality directly affects retention.
- Customers are segmented into **low / medium / high** churn-risk tiers using `churn_score`, which can drive targeted retention campaigns.
